# fes-bench：表格与作图

从 `fes-bench` 仓库读取 `result/tables/*.csv` 与 `data/processed/`，生成文档里的表 2a–6 和图，导出为可直接粘贴进文档的 Excel 与 Markdown。

**本 notebook 只做展示，不重新计算实验。** 技能分与斜率换算均直接读取仓库 CSV；notebook 只做展示与单位换算。

| 文档表 | 来源 CSV |
|---|---|
| 表 2a 体系与真值级别 | `system_inventory.csv` |
| 表 2b 相与结构 | `phase_inventory.csv` |
| 表 3 切分 | `split_definitions.csv` |
| 表 4 指标定义 | `metric_definitions.csv` |
| 表 5a / 5b / 5c 主结果 | `predictor_comparison_{temp_extrap,phase_lopo,system_loso}.csv` |
| 表 6 逐对相 $T_c$ 误差 | `crossing_errors_temp_extrap.csv` |

**用法**：改第一个代码单元的 `SOURCE`，然后 Runtime → Run all。最后一个单元会打包下载 `fes_bench_outputs.zip`。

In [ ]:
# ===== 配置 =====
SOURCE = "github"          # "github" 克隆仓库 | "upload" 上传 zip | "local" 已有目录
REPO_URL = "https://github.com/zirenjin/fes-bench.git"
BRANCH = ""                # 留空 = 默认分支
LOCAL_ROOT = "fes-bench"   # 仓库所在目录（local 模式直接用）
OUT_DIR = "fes_bench_outputs"

INCLUDE_PLACEHOLDER_ROWS = True   # 主表中为尚未运行的 predictor 留空行（TIP、E+F_QH 等）
SHOW_FIXTURE_ROWS = False         # 评测器自检用的 reference / 噪声 / 常数偏移行，默认不进主表

In [ ]:
# ===== 获取数据 =====
import os, subprocess, zipfile, pathlib, shutil, hashlib, json

def _run(cmd):
    print("$", " ".join(cmd)); subprocess.run(cmd, check=True)

root = pathlib.Path(LOCAL_ROOT)
if SOURCE == "github":
    if (root / ".git").exists():
        _run(["git", "-C", str(root), "pull", "-q"])
    else:
        cmd = ["git", "clone", "-q", "--depth", "1"] + (["-b", BRANCH] if BRANCH else []) + [REPO_URL, str(root)]
        _run(cmd)
elif SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()          # 上传仓库 zip（至少包含 result/tables 和 data/processed）
    name = next(iter(uploaded))
    shutil.rmtree(root, ignore_errors=True); root.mkdir(parents=True)
    with zipfile.ZipFile(name) as z: z.extractall(root)
    hits = list(root.rglob("result/tables"))
    if not hits: raise FileNotFoundError("zip 里找不到 result/tables/")
    root = hits[0].parent.parent       # zip 里多一层目录也能找到
elif SOURCE != "local":
    raise ValueError(SOURCE)

TABLES = root / "result" / "tables"
DATA = root / "data" / "processed"
try:
    COMMIT = subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                            capture_output=True, text=True, check=True).stdout.strip()
except Exception:
    COMMIT = "unknown"
print("repo root:", root.resolve(), "| commit:", COMMIT)
print("tables:", sorted(p.name for p in TABLES.glob("*.csv")))

In [ ]:
# ===== 读取与通用工具 =====
import numpy as np, pandas as pd
pd.set_option("display.max_colwidth", 80)

def read_csv(name):
    # 全部按字符串读，保留 n/a:* 标记与空值的区别
    return pd.read_csv(TABLES / f"{name}.csv", dtype=str, keep_default_na=False)

def read_meta(name):
    p = TABLES / f"{name}.meta.json"
    return json.loads(p.read_text()) if p.exists() else {}

RAW = {p.stem: read_csv(p.stem) for p in sorted(TABLES.glob("*.csv"))}
META = {k: read_meta(k) for k in RAW}
SPLITS = ["temp_extrap", "phase_lopo", "system_loso"]
SPLIT_LABEL = {"temp_extrap": "表5a 温区外推", "phase_lopo": "表5b 留一相", "system_loso": "表5c 留一体系"}

NA_LABEL = {
    "n/a:no_reference_crossing": "n/a（无参照交点）",
    "n/a:pair_only_predictor":   "n/a（仅预测ΔG）",
    "n/a:no_training_phase":     "n/a（无训练相）",
    "n/a:degenerate_prediction": "n/a（退化预测）",
    "n/a:not_trained_for_split": "n/a（未按该切分训练）",
    "n/a:input_unavailable": "n/a（输入不可得）",
    "n/a:no_seed": "n/a（无 seed）",
}
MISSING = "—"   # 空值 = 尚未计算，不是结构性不适用

def num(x):
    try: return float(x)
    except (TypeError, ValueError): return np.nan

def is_marker(x): return isinstance(x, str) and x.startswith("n/a:")

def fmt(x, scale=1.0, digits=2, signed=False):
    # 数值格式化；n/a 标记转中文；空值转 —
    if is_marker(x): return NA_LABEL.get(x, x)
    v = num(x)
    if np.isnan(v): return MISSING
    v *= scale
    return f"{v:+.{digits}f}" if signed else f"{v:.{digits}f}"

def sha256(p):
    return hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()

# predictor 显示名、分组、顺序。未登记的 predictor 放到"其他"组，不会被静默丢掉
PREDICTORS = {
    "zero":                ("zero",                          "平凡地板", 10),
    "constant_delta_g":    ("constant ΔG",                   "平凡地板", 11),
    "global_mean_delta_g": ("global-mean ΔG",                "平凡地板", 12),
    "interp_const":        ("interp_constant",               "平凡地板", 13),
    "phase_id_mlp":        ("phase_id_mlp",                  "平凡地板", 14),
    "bartel2018":          ("Bartel descriptor",             "外部方法", 20),
    "tip_uma":             ("TIP[UMA]†",                     "外部方法", 21),
    "qh_only":             ("E + F_QH",                      "受控消融", 30),
    "qh_residual":         ("E + F_QH + r_θ",                "受控消融", 31),
    "repr_only":           ("representation regression",     "受控消融", 32),
    "repr. regression (polynomial)": ("repr. regression (polynomial)", "受控消融", 33),
    "repr. regression (tlog)": ("repr. regression (tlog)", "受控消融", 34),
}
PLACEHOLDERS = ["tip_uma", "qh_only", "qh_residual", "repr_only"]
FIXTURE_PREFIXES = ("reference", "root_stability", "constant_offset")
GROUP_ORDER = ["平凡地板", "外部方法", "受控消融", "其他", "评测器自检"]

def pinfo(key):
    if key.startswith(FIXTURE_PREFIXES): return (key, "评测器自检", 90)
    return PREDICTORS.get(key, (key, "其他", 80))

print({k: v.shape for k, v in RAW.items()})

## 一致性检查

**先看这一格再填表。** 这里自动检查 CSV 之间是否自洽。出现 ⚠ 的项，对应表格里的数字不应直接进文档，要先回仓库修表脚本。

In [ ]:
# ===== 一致性检查 =====
CHECKS = []
def check(ok, msg, level="WARN"):
    tag = "✓" if ok else ("⚠" if level == "WARN" else "ℹ")
    CHECKS.append(f"{tag} {msg}"); print(CHECKS[-1])

pc = {s: RAW.get(f"predictor_comparison_{s}") for s in SPLITS}
metric_cols = ["delta_G_MAE_eV_per_atom", "sign_accuracy", "Tc_error_K"]

# 1. 主表是否真的随切分变化
for p in sorted(set().union(*[set(df.predictor) for df in pc.values() if df is not None])):
    rows = [tuple(df.loc[df.predictor == p, metric_cols].iloc[0]) for df in pc.values()
            if df is not None and (df.predictor == p).any()]
    vals = [r for r in rows if any(num(x) == num(x) for x in r)]
    if len(vals) >= 2 and len(set(vals)) == 1 and not p.startswith(FIXTURE_PREFIXES):
        check(False, f"{p}: 三个切分的 ΔG MAE / 符号 / Tc 完全相同 —— 主表可能没有按切分评测")

# 2. LOPO/LOSO 上 constant ΔG 结构性不可拟合，却有数
for s in ["phase_lopo", "system_loso"]:
    df = pc[s]
    if df is not None and (df.predictor == "constant_delta_g").any():
        r = df[df.predictor == "constant_delta_g"].iloc[0]
        z = df[df.predictor == "zero"].iloc[0] if (df.predictor == "zero").any() else None
        has = not np.isnan(num(r.delta_G_MAE_eV_per_atom))
        same_as_zero = z is not None and tuple(r[metric_cols]) == tuple(z[metric_cols])
        check(not has, f"{s}: constant ΔG 有数值（应为 n/a:no_training_phase）"
              + ("，且与 zero 行完全相同" if same_as_zero else ""))

# 3. zero 预测 ΔG≡0，交点无定义
for s, df in pc.items():
    if df is not None and (df.predictor == "zero").any():
        v = num(df[df.predictor == "zero"].iloc[0].Tc_error_K)
        check(np.isnan(v), f"{s}: zero 行 Tc 误差 = {v:.1f} K（ΔG≡0 时求根退化，应为 n/a）")

# 4. 主表缺的列
need = {"ranking_accuracy": "相序正确率", "skill_score": "技能分", "Tc_err_from_dG_K": "斜率换算",
        "seed_std": "seed 散布", "floor_predictor": "技能分分母"}
for s, df in pc.items():
    if df is not None:
        miss = [v for k, v in need.items() if k not in df.columns]
        check(not miss, f"{s}: 主表 CSV 缺列 {miss}（notebook 中技能分与斜率换算会派生，其余留 —）", "INFO")

# 5. 受控消融行是否存在
present = set().union(*[set(df.predictor) for df in pc.values() if df is not None])
check(bool(present & {"qh_only", "qh_residual", "repr_only", "polynomial", "tlog_polynomial"}),
      "主表里没有任何受控消融行（E1 的 polynomial / tlog 检查点未接入主表）", "INFO")

# 6. qh_reliable 计数与逐相清单是否一致
si, ph = RAW.get("system_inventory"), RAW.get("phase_inventory")
if si is not None and ph is not None:
    for _, r in si.iterrows():
        sub = ph[ph.system == r.system]
        n_true = (sub.qh_reliable.str.lower() == "true").sum()
        n_blank = (sub.qh_reliable == "").sum()
        check(str(n_true) == r.qh_reliable_count and n_blank == 0,
              f"{r.system}: system_inventory 记 qh_reliable_count={r.qh_reliable_count}，"
              f"phase_inventory 中 true={n_true}、空={n_blank}")
    for _, r in si.iterrows():
        if "D3" in r.truth_level and "D3" not in r.functional:
            check(False, f"{r.system}: functional 写 '{r.functional}'，但真值级别是 '{r.truth_level}'")

# 7. temp_extrap 的交点必须全在测试区
sd = RAW.get("split_definitions")
if sd is not None:
    te = sd[sd.split == "temp_extrap"]
    bad = [x for c in te.crossing_in_test for x in c.split(";") if x and not x.endswith("=test")]
    check(not bad, f"temp_extrap 交点全部落在测试区" + (f"：例外 {bad}" if bad else ""))

# 8. meta.json 中记录的缺失原因汇总
for k, m in META.items():
    miss = m.get("missing", [])
    if miss:
        reasons = pd.Series([x.get("reason", "?") if isinstance(x, dict) else str(x) for x in miss]).value_counts()
        check(True, f"{k}: meta 记录 {len(miss)} 处缺失 → " + "; ".join(f"{i}×{n}" for i, n in reasons.items()), "INFO")

n_warn = sum(c.startswith("⚠") for c in CHECKS)
print(f"\n共 {n_warn} 项警告。")

## 表 2a 体系与真值级别

In [ ]:
def table_2a():
    df = RAW["system_inventory"]
    out = pd.DataFrame({
        "体系": df.system,
        "相": df.phases.str.replace(";", ", "),
        "相数": df.n_phases,
        "参照方法": df.truth_level,
        "泛函": df.functional,
        "T 范围 (K)": [f"{fmt(a,digits=0)}–{fmt(b,digits=0)}" for a, b in zip(df.T_min_K, df.T_max_K)],
        "网格点": df.grid_points,
        "ΔG std (meV/atom)": [fmt(a, digits=2) if a == b else f"{fmt(a,digits=2)}–{fmt(b,digits=2)}"
                               for a, b in zip(df.delta_G_std_min_meV_per_atom, df.delta_G_std_max_meV_per_atom)],
        "相对数": df.relative_numbers,
        "QH 可用": [f"{c}/{n}" for c, n in zip(df.qh_reliable_count, df.n_phases)],
        "DOI": df.doi,
    })
    return out
T2A = table_2a(); T2A

## 表 2b 相与结构

In [ ]:
def table_2b():
    df = RAW["phase_inventory"]
    src = df.representative_source.map(lambda s: pathlib.PurePath(s).name if s else MISSING)
    return pd.DataFrame({
        "体系": df.system, "相": df.phase,
        "空间群（弛豫前→后）": [f"{a or '?'} → {b or '?'}" for a, b in zip(df.space_group_before, df.space_group_after)],
        "原胞原子数": df.primitive_atoms,
        "代表结构来源": src,
        "QH 超胞": df.qh_supercell, "最短边 (Å)": df.shortest_edge_A.map(lambda x: fmt(x, digits=2)),
        "q 网格": df.q_mesh,
        "虚频占比": df.imaginary_fraction.map(lambda x: fmt(x, scale=100, digits=2) + ("%" if num(x) == num(x) else "")),
        "qh_reliable": df.qh_reliable.replace("", MISSING),
    })
T2B = table_2b(); T2B

## 表 3 切分

In [ ]:
def table_3():
    df = RAW["split_definitions"]
    def crossings(s):
        parts = [x for x in s.split(";") if x]
        return f"{sum(p.endswith('=test') for p in parts)}/{len(parts)} 在测试区"
    return pd.DataFrame({
        "切分": df.split, "折": df.fold, "留出单位": df.held_out_unit,
        "训练帧": df.train_frames, "测试帧": df.test_frames,
        "参照交点": df.crossing_in_test.map(crossings),
        "SHA-256": df.sha256.str[:10], "commit": df.commit.str[:7],
    })
T3 = table_3(); T3

## 表 4 指标定义

In [ ]:
def table_4():
    df = RAW["metric_definitions"].copy()
    df["来源"] = "仓库"
    extra = pd.DataFrame([
        ("skill_score", "1 − MAE_model / MAE_floor；floor 优先取 constant ΔG，不可用时取 global-mean ΔG", "—", "↑", "notebook 派生（待入库）"),
        ("ranking_accuracy", "全部相的预测排序与参照完全一致的网格点占比", "fraction", "↑", "未实现"),
        ("Tc_err_from_dG", "交点处 |ΔG 误差| / |dΔG/dT|，把能量误差按斜率换算成温度", "K", "↓", "仓库（在 crossing 表中）"),
    ], columns=["metric", "definition", "unit", "direction", "来源"])
    df = pd.concat([df, extra[~extra.metric.isin(df.metric)]], ignore_index=True)
    return df.rename(columns={"metric": "指标", "definition": "定义", "unit": "单位", "direction": "方向"})
T4 = table_4(); T4

## 表 5a / 5b / 5c 主结果

行按「平凡地板 → 外部方法 → 受控消融」分组。**技能分在这里派生**：`1 − MAE/MAE_floor`，分母写在「技能分分母」列，不同切分的分母可能不同。斜率换算取 crossing 表中该 predictor 有交点的对的均值。

In [ ]:
def table_5(split):
    df = RAW[f"predictor_comparison_{split}"].copy()
    if not SHOW_FIXTURE_ROWS:
        df = df[~df.predictor.str.startswith(FIXTURE_PREFIXES)]
    if INCLUDE_PLACEHOLDER_ROWS:
        for p in PLACEHOLDERS:
            if p not in set(df.predictor):
                df.loc[len(df)] = {c: "" for c in df.columns} | {"predictor": p}
    rows = []
    for _, r in df.iterrows():
        name, group, order = pinfo(r.predictor)
        rows.append({"_order": order, "分组": group, "predictor": name,
            "ΔG MAE ↓ (meV/atom)": fmt(r.delta_G_MAE_eV_per_atom, scale=1000, digits=2),
            "技能分 ↑": fmt(r.get("skill_score", ""), digits=2, signed=True),
            "技能分分母": fmt(r.get("floor_predictor", "")),
            "符号正确率 ↑ (%)": fmt(r.sign_accuracy, scale=100, digits=1),
            "相序正确率 ↑ (%)": fmt(r.get("ranking_accuracy", ""), scale=100, digits=1),
            "T_c 误差 ↓ (K)": fmt(r.Tc_error_K, digits=1),
            "斜率换算 ↓ (K)": fmt(r.get("Tc_err_from_dG_K", ""), digits=0),
            "假/漏交点 ↓": fmt(r.get("false_crossings", "")) + "/" + fmt(r.get("missed_crossings", "")),
            "seed 散布": fmt(r.get("seed_std", ""), digits=2),
            "_skill": num(r.get("skill_score", "")),})
    out = pd.DataFrame(rows); out["_g"] = out["分组"].map(GROUP_ORDER.index)
    return out.sort_values(["_g", "_order"]).drop(columns=["_g", "_order"]).reset_index(drop=True), "技能分与分母直接读取主表 CSV"

T5 = {}
for s in SPLITS:
    t, note = table_5(s); T5[s] = (t, note)
    print(f"\n{SPLIT_LABEL[s]} —— {note}")
    display(t.drop(columns=["_skill"]))

## 表 6 逐对相 $T_c$ 误差（temp_extrap）

In [ ]:
def table_6(split="temp_extrap"):
    ce = RAW[f"crossing_errors_{split}"].copy()
    def pick_ref(s):   # 优先数值；否则取 no_reference_crossing 标记
        nums = [x for x in s if not np.isnan(num(x))]
        if nums: return nums[0]
        return next((x for x in s if x == "n/a:no_reference_crossing"), next(iter(s), ""))
    ref = ce.groupby("pair").agg(ref=("reference_Tc_K", pick_ref),
                                 slope=("crossing_slope_eV_per_atom_per_K",
                                        lambda s: next((x for x in s if num(x) > 0), "")))
    def cell(r):
        if is_marker(r.reference_Tc_K): 
            f = num(r.false_crossings)
            return NA_LABEL.get(r.reference_Tc_K) + (f" +{int(f)}假" if f > 0 else "")
        if num(r.missed_crossings) > 0: return "漏"
        v = fmt(r.Tc_error_K, digits=0, signed=True)
        f = num(r.false_crossings)
        return v + (f" +{int(f)}假" if f > 0 else "")
    if not SHOW_FIXTURE_ROWS:   # 参照 T_c 与斜率已在上面从全部行（含 reference）取得，再过滤自检行
        ce = ce[~ce.predictor.str.startswith(FIXTURE_PREFIXES)]
    ce["cell"] = ce.apply(cell, axis=1)
    piv = ce.pivot_table(index="pair", columns="predictor", values="cell", aggfunc="first")
    order = sorted(piv.columns, key=lambda p: (GROUP_ORDER.index(pinfo(p)[1]), pinfo(p)[2]))
    piv = piv[order].rename(columns=lambda p: pinfo(p)[0])
    piv.insert(0, "交点斜率 dΔG/dT (meV/atom/K)", ref.slope.map(lambda x: fmt(x, scale=1000, digits=4)))
    piv.insert(0, "参照 T_c (K)", ref.ref.map(lambda x: fmt(x, digits=1)))
    piv.index = [p.replace("_minus_", " − ") for p in piv.index]
    return piv.reset_index().rename(columns={"index": "相对"})

T6 = table_6(); T6

## 图

- **图 A**：参照 ΔG(T) 与交点，灰带为 temp_extrap 训练区。直接展示目标量只有几 meV/atom。
- **图 B**：ΔG MAE vs 符号正确率（每切分一张）。竖线为参照 ΔG 的标准差——MAE 落在这条线附近，就和不预测差不多。
- **图 C**：技能分，零线为地板。
- **图 D**：各 predictor 在有交点的对上的 $T_c$ 误差，x 为漏交点。

In [ ]:
import itertools, matplotlib.pyplot as plt
FIG = pathlib.Path(OUT_DIR) / "figures"; FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
import matplotlib.font_manager as fm
def _cjk():
    return next((x.name for x in fm.fontManager.ttflist
                 if any(k in x.name for k in ["Noto Sans CJK", "SimHei", "WenQuanYi", "PingFang", "Microsoft YaHei"])), None)
if _cjk() is None:   # Colab 默认没有中文字体，装一次
    try:
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-noto-cjk"], check=True,
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        for f in pathlib.Path("/usr/share/fonts").rglob("NotoSansCJK*.tt*"): fm.fontManager.addfont(str(f))
    except Exception as e:
        print("中文字体安装失败，图中文字可能显示为方框：", e)
if _cjk(): plt.rcParams["font.sans-serif"] = [_cjk()] + plt.rcParams["font.sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

def save(fig, name):
    for ext in ("png", "pdf"): fig.savefig(FIG / f"{name}.{ext}", bbox_inches="tight")
    plt.show()

# ---- 图 A：参照 ΔG(T) ----
thr = {}
tsplit = DATA / "splits" / "temp_extrap.json"
if tsplit.exists(): thr = json.loads(tsplit.read_text()).get("threshold_T_K", {})
systems = [d.name for d in sorted(DATA.iterdir()) if (d / "system.json").exists()]
fig, axes = plt.subplots(1, len(systems), figsize=(3.4 * len(systems), 2.8), squeeze=False)
for ax, sysn in zip(axes[0], systems):
    phases = json.loads((DATA / sysn / "system.json").read_text())["phases"]
    G = {p: pd.read_csv(DATA / sysn / p / "reference_G.csv") for p in phases
         if (DATA / sysn / p / "reference_G.csv").exists()}
    for a, b in itertools.combinations(G, 2):
        m = G[a].merge(G[b], on="T_K", suffixes=("_a", "_b"))
        d = (m.G_eV_per_atom_a - m.G_eV_per_atom_b) * 1000
        ax.plot(m.T_K, d, lw=1.4, label=f"{a} − {b}")
        # 找变号点；参照表有量化，允许恰好为 0 的点（取非零邻居判断变号）
        v = d.values; nz = np.flatnonzero(v != 0)
        for i, j in zip(nz[:-1], nz[1:]):
            if np.sign(v[i]) != np.sign(v[j]):
                t = m.T_K.iloc[i] - v[i] * (m.T_K.iloc[j] - m.T_K.iloc[i]) / (v[j] - v[i])
                ax.plot(t, 0, "ko", ms=4)
    if sysn in thr: ax.axvspan(ax.get_xlim()[0], thr[sysn], color="0.9", zorder=0)
    ax.axhline(0, color="0.4", lw=0.7); ax.set_title(sysn); ax.set_xlabel("T (K)")
    ax.legend(fontsize=6, frameon=False)
axes[0][0].set_ylabel("参照 ΔG (meV/atom)")
fig.tight_layout(); save(fig, "A_reference_delta_g")

In [ ]:
# ---- 图 B：MAE vs 符号正确率 ----
ref_std = {}
si = RAW.get("system_inventory")
if si is not None:
    v = pd.to_numeric(si.delta_G_std_max_meV_per_atom, errors="coerce")
    ref_std = {"median": float(v.median())}
COL = {"平凡地板": "tab:gray", "外部方法": "tab:orange", "受控消融": "tab:blue", "其他": "tab:purple"}
fig, axes = plt.subplots(1, len(SPLITS), figsize=(3.6 * len(SPLITS), 3.0), sharey=True)
for ax, s in zip(np.atleast_1d(axes), SPLITS):
    df = RAW[f"predictor_comparison_{s}"]
    df = df[~df.predictor.str.startswith(FIXTURE_PREFIXES)]
    for _, r in df.iterrows():
        x, y = num(r.delta_G_MAE_eV_per_atom) * 1000, num(r.sign_accuracy) * 100
        if np.isnan(x) or np.isnan(y): continue
        name, group, _ = pinfo(r.predictor)
        ax.scatter(x, y, color=COL.get(group, "k"), s=28)
        ax.annotate(name, (x, y), fontsize=6, xytext=(3, 3), textcoords="offset points")
    if ref_std: ax.axvline(ref_std["median"], ls="--", color="0.5", lw=0.8)
    ax.set_title(SPLIT_LABEL[s]); ax.set_xlabel("ΔG MAE (meV/atom)")
np.atleast_1d(axes)[0].set_ylabel("符号正确率 (%)")
fig.tight_layout(); save(fig, "B_mae_vs_sign")

# ---- 图 C：技能分 ----
fig, axes = plt.subplots(1, len(SPLITS), figsize=(3.8 * len(SPLITS), 3.0))
for ax, s in zip(np.atleast_1d(axes), SPLITS):
    t, note = T5[s]; t = t[t._skill.notna()]
    ax.barh(t.predictor, t._skill, color=[COL.get(g, "k") for g in t["分组"]])
    ax.axvline(0, color="k", lw=0.8); ax.invert_yaxis()
    ax.set_title(SPLIT_LABEL[s]); ax.set_xlabel("技能分"); ax.text(0.02, -0.25, note, transform=ax.transAxes, fontsize=6)
fig.tight_layout(); save(fig, "C_skill_score")

# ---- 图 D：T_c 误差 ----
ce = RAW["crossing_errors_temp_extrap"]
ce = ce[~ce.predictor.str.startswith(FIXTURE_PREFIXES) & ce.reference_Tc_K.map(lambda x: not is_marker(x))]
pairs = sorted(ce.pair.unique()); preds = sorted(ce.predictor.unique(), key=lambda p: (GROUP_ORDER.index(pinfo(p)[1]), pinfo(p)[2]))
fig, ax = plt.subplots(figsize=(1.2 + 1.1 * len(pairs), 3.0))
w = 0.8 / max(len(preds), 1)
for j, p in enumerate(preds):
    for i, pair in enumerate(pairs):
        r = ce[(ce.predictor == p) & (ce.pair == pair)]
        if r.empty: continue
        r = r.iloc[0]; x = i + (j - len(preds) / 2) * w + w / 2
        if num(r.missed_crossings) > 0: ax.plot(x, 0, "x", color=COL.get(pinfo(p)[1], "k"), ms=6)
        elif not np.isnan(num(r.Tc_error_K)): ax.bar(x, num(r.Tc_error_K), w, color=COL.get(pinfo(p)[1], "k"))
    ax.bar(0, 0, 0, color=COL.get(pinfo(p)[1], "k"), label=pinfo(p)[0])
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(range(len(pairs))); ax.set_xticklabels([p.replace("_minus_", "\n− ") for p in pairs], fontsize=6)
ax.set_ylabel("T_c 误差 (K)"); ax.set_title("温区外推：逐对相 T_c 误差（x = 漏交点）")
ax.legend(fontsize=6, frameon=False, bbox_to_anchor=(1, 1))
fig.tight_layout(); save(fig, "D_tc_errors")

## 导出

- `tables_for_doc.xlsx`：每张表一个工作表（表2a … 表6）。在 Excel / 飞书表格里选中区域复制，粘贴到文档表格即可。
- `md/*.md`：同样内容的 Markdown 表格。
- `checks.txt`：上面的一致性检查结果。**有警告时请连同表格一起看。**
- `manifest.json`：数据 commit 与每个输入 CSV 的 SHA-256，保证文档里的数字能追溯回仓库版本。

In [ ]:
out = pathlib.Path(OUT_DIR); (out / "md").mkdir(parents=True, exist_ok=True)
SHEETS = {"表2a": T2A, "表2b": T2B, "表3": T3, "表4": T4,
          "表5a": T5["temp_extrap"][0], "表5b": T5["phase_lopo"][0], "表5c": T5["system_loso"][0], "表6": T6}
NOTES = {"表5a": T5["temp_extrap"][1], "表5b": T5["phase_lopo"][1], "表5c": T5["system_loso"][1]}

def to_md(df):
    df = df.astype(str)
    head = "| " + " | ".join(df.columns) + " |\n|" + "---|" * len(df.columns) + "\n"
    return head + "\n".join("| " + " | ".join(r) + " |" for r in df.values)

with pd.ExcelWriter(out / "tables_for_doc.xlsx") as xw:
    for name, df in SHEETS.items():
        df = df.drop(columns=[c for c in df.columns if c.startswith("_")])
        df.to_excel(xw, sheet_name=name, index=False)
        text = f"# {name}\n\n" + (f"{NOTES[name]}\n\n" if name in NOTES else "") + to_md(df) + "\n"
        (out / "md" / f"{name}.md").write_text(text, encoding="utf-8")

(out / "checks.txt").write_text("\n".join(CHECKS) + "\n", encoding="utf-8")
manifest = {"commit": COMMIT, "source": SOURCE,
            "inputs": {p.name: sha256(p) for p in sorted(TABLES.glob("*.csv"))},
            "warnings": sum(c.startswith("⚠") for c in CHECKS)}
(out / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False))

shutil.make_archive(OUT_DIR, "zip", out)
print("written:", sorted(str(p.relative_to(out)) for p in out.rglob("*") if p.is_file()))
try:
    from google.colab import files; files.download(f"{OUT_DIR}.zip")
except ImportError:
    print(f"not in Colab; archive at {OUT_DIR}.zip")